In [ ]:
import torch
import sys

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
    print("\n✅ GPU READY FOR TRAINING!")

    # A100 optimizations
    device = torch.device('cuda')
    if 'A100' in torch.cuda.get_device_name(0):
        print("\n🚀 A100 DETECTED! Enabling optimizations...")
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("✅ TF32 enabled for faster training")
else:
    print("\n❌ NO GPU AVAILABLE! Go to Runtime → Change runtime type → GPU")
    device = torch.device('cpu')

## 📦 Step 2: Install Required Libraries

In [ ]:
# Install transformers for ViT
!pip install -q transformers timm gradio scikit-learn albumentations

## 💾 Step 3: Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content')

# Clone repository
!git clone -q https://github.com/Nishvaraj/multimodal-emotion-recognition.git
os.chdir('/content/multimodal-emotion-recognition')

print("✅ Repository cloned!")
print("Current directory:", os.getcwd())

## 📊 Step 4: Verify Datasets

In [ ]:
from pathlib import Path
import sys

# Check FER2013 dataset
fer2013_path = Path('data/raw/fer2013')
ravdess_path = Path('data/raw/ravdess')

print("🔍 Checking datasets...\n")

if fer2013_path.exists():
    train_images = list((fer2013_path / 'train').glob('*/*'))
    test_images = list((fer2013_path / 'test').glob('*/*'))
    print(f"✅ FER2013 Found:")
    print(f"   Training images: {len(train_images)}")
    print(f"   Test images: {len(test_images)}")
    print(f"   Total: {len(train_images) + len(test_images)} images")
else:
    print(f"❌ FER2013 not found at {fer2013_path}")
    print("   Ensure data/raw/fer2013/train and /test directories exist")

print()

if ravdess_path.exists():
    audio_files = list(ravdess_path.glob('*.wav'))
    print(f"✅ RAVDESS Found:")
    print(f"   Audio files: {len(audio_files)}")
else:
    print(f"❌ RAVDESS not found at {ravdess_path}")

## 🎯 Step 5: Load Data Loader Module

In [ ]:
# Add backend to path
sys.path.insert(0, '/content/multimodal-emotion-recognition/backend/services')

# Import custom data loader
from data_loader import FER2013Dataset, create_dataloaders

print("✅ Data loader imported successfully!")

# Test loading a batch
print("\n📦 Creating test batch...")
train_loader, test_loader = create_dataloaders(
    train_dir='data/raw/fer2013/train',
    test_dir='data/raw/fer2013/test',
    batch_size=32
)

# Get first batch
images, labels = next(iter(train_loader))
print(f"✅ Batch loaded!")
print(f"   Images shape: {images.shape}")
print(f"   Labels shape: {labels.shape}")
print(f"   Image range: [{images.min():.2f}, {images.max():.2f}]")

## 📊 Step 6: Data Statistics & Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Emotion mapping
emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# Count labels in training set
all_labels = []
for _, labels in train_loader:
    all_labels.extend(labels.cpu().numpy())

label_counts = Counter(all_labels)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
counts = [label_counts[i] for i in range(7)]
ax.bar(emotions, counts, color='skyblue', edgecolor='navy')
ax.set_xlabel('Emotion', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('FER2013 Training Set - Class Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n📊 Dataset Statistics:")
for i, emotion in enumerate(emotions):
    print(f"   {emotion}: {counts[i]} images")
print(f"   Total: {sum(counts)} images")

## 🎨 Step 7: Visualize Sample Images

In [ ]:
# Get a batch
images, labels = next(iter(train_loader))

# Plot 16 random samples
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.ravel()

for idx in range(16):
    # Denormalize image
    img = images[idx].cpu().numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)

    axes[idx].imshow(img)
    axes[idx].set_title(f"{emotions[labels[idx].item()]}", fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Sample Images from FER2013 Training Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Ready for Phase 2 training!")

## 📝 Summary

✅ GPU verified and optimized
✅ Repository cloned
✅ FER2013 dataset loaded
✅ DataLoader created
✅ Data visualized

**Next**: Run `PHASE2_COLAB_02_ResNet_Baseline.ipynb` to train baseline CNN